In [2]:
# !pip install ragas

In [12]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import faiss
import json
from pathlib import Path
from ollama import embed,chat

from rank_bm25 import BM25Okapi
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sentence_transformers import CrossEncoder

import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('punkt')

stop_words = set(stopwords.words('english')) ##set the stop words

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [4]:
ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled")

In [5]:
df = pd.DataFrame(ds['train'])  ### creating dataframe from ds

In [37]:
class full_rag:

    def __init__(self, df, query):
        self.df           = df
        self.query        = query
        self.index_path   = r'C:\Users\ADMIN\MLops\ollama\data\processed\medical_rag.index'
        self.context_list = self.get_context_in_list(df)

        # BM25 — built once ✅
        tokenized = [
            self.tokenize(c)
            for c in self.context_list
        ]
        self.bm25 = BM25Okapi(tokenized)

        # FAISS — loaded once ✅
        self.index = faiss.read_index(self.index_path)

        # reranker — loaded once ✅
        self.reranker = CrossEncoder(
            'cross-encoder/ms-marco-MiniLM-L-6-v2',
            device='cuda'
        )

    def get_context_in_list(self, df: pd.DataFrame) -> list:
        context_list = []
        for _, row in df.iterrows():
            for cntxt in row['context']['contexts']:
                if len(cntxt.split()) > 20:
                    context_list.append(cntxt)
        return context_list

    def fetch_and_save_embeddings(self,
            max_neighbors: int = 32,
            ef_construction: int = 200,
            efsearch: int = 50):

        cntxt_emb = np.array(
            embed(
                model='embeddinggemma',
                input=self.context_list
            )['embeddings'],
            dtype=np.float32
        )
        faiss.normalize_L2(cntxt_emb)
        dim   = cntxt_emb.shape[1]
        index = faiss.IndexHNSWFlat(
            dim, max_neighbors,
            faiss.METRIC_INNER_PRODUCT
        )
        index.hnsw.efConstruction = ef_construction
        index.hnsw.efSearch       = efsearch
        index.add(cntxt_emb)
        faiss.write_index(index, self.index_path)
        print("index saved ✅")

    def tokenize(self, text: str):
        text   = text.lower()
        tokens = re.findall(r'\b[a-z]+\b', text)
        tokens = [t for t in tokens
                  if t not in stop_words]
        return tokens

    def hybrid_search(self, k: int = 60,
                      top_k: int = 50):

        # BM25 search
        query_tokens = self.tokenize(self.query)
        bm25_scores  = self.bm25.get_scores(query_tokens)
        bm25_ranked  = np.argsort(bm25_scores)[::-1]

        # vector search
        q_emb = np.array(
            embed(
                model='embeddinggemma',
                input=[self.query]
            )['embeddings'],
            dtype=np.float32
        )
        faiss.normalize_L2(q_emb)
        _, vec_indices = self.index.search(
            q_emb, len(self.context_list)
        )
        vec_ranked = vec_indices[0]   # ← indices not scores

        # RRF fusion
        rrf_scores = {}

        for rank, idx in enumerate(bm25_ranked):  # ✅
            rrf_scores[idx] = \
                rrf_scores.get(idx, 0) + 1/(rank + k)

        for rank, idx in enumerate(vec_ranked):   # ✅
            rrf_scores[idx] = \
                rrf_scores.get(idx, 0) + 1/(rank + k)

        sorted_idx = sorted(
            rrf_scores,
            key=lambda x: rrf_scores[x],
            reverse=True
        )[:top_k]

        return [
            {
                "chunk"    : self.context_list[idx],
                "rrf_score": round(rrf_scores[idx], 5),
                "idx"      : idx
            }
            for idx in sorted_idx
        ]

    def rerank(self, top_k: int = 3):

        candidates = self.hybrid_search()  # ✅ no param

        pairs = [
            [self.query, c['chunk']]
            for c in candidates
        ]
        scores = self.reranker.predict(pairs)  # ✅ self.reranker

        for i, c in enumerate(candidates):
            c['rerank_score'] = float(scores[i])

        reranked = sorted(
            candidates,
            key=lambda x: x['rerank_score'],
            reverse=True
        )
        return reranked[:top_k]

    def get_relevance(self):

        reranked  = self.rerank()
        THRESHOLD = 0.0
        relevant  = [
            r for r in reranked
            if r['rerank_score'] > THRESHOLD
        ]
        if not relevant:
            return {
                "question": self.query,
                "answer"  : "No relevant context found.",
                "contexts": []
            }
        return relevant
    
    def parse_answer(self,response: str) -> str:
        try:
            parsed = json.loads(response)
            # get whatever key exists
            return list(parsed.values())[0]
        except:
            pass
        try:
            match = re.search(
                r'```json\s*(.*?)\s*```',
                response, re.DOTALL
            )
            if match:
                parsed = json.loads(match.group(1))
                return list(parsed.values())[0]
        except:
            pass
        return response

    def generate(self) -> dict:

        relevant = self.get_relevance()

        if isinstance(relevant, dict):
            return relevant

        contexts    = [r['chunk'] for r in relevant]
        context_str = "\n\n".join(
            [f"[{i+1}] {c}"
             for i, c in enumerate(contexts)]
        )
        prompt = f"""You are a medical AI assistant.
                Answer using ONLY the context below.
                Reply in plain text, no JSON no markdown.
                If context insufficient say so clearly.

                Context:
                {context_str}

                Question: {self.query}

                Answer:"""
        

        response =chat(
        model='gemma4:latest',
        messages=[{'role': 'user', 'content':prompt}],
        stream=False,
        think = False

        )
        answer = self.parse_answer(
            response["message"]["content"]
        )
        return {
            "question": self.query,
            "answer"  : answer,
            "contexts": contexts
        }

In [38]:
query = "what is programmed cell death mechanism?"
RAG_END_TO_END = full_rag(df=df, query = query)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3683.89it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
result = RAG_END_TO_END.generate()
print(result['question'])
print(result['answer'])
print(len(result['contexts']), "contexts used")

what is programmed cell death mechanism?
Programmed cell death (PCD) is the regulated death of cells within an organism.
1 contexts used


### Step -2 Get evaluation done now

In [41]:
# !pip install ragas langsmith

In [73]:
test_questions = [i for i in df.iloc[:10]['question']]

print(f"total test questions: {len(test_questions)}")

total test questions: 10


In [74]:
eval_data = []

for i, q in enumerate(test_questions):
    print(f"processing {i+1}/{len(test_questions)}: {q[:50]}")
    
    rag    = full_rag(df, q)
    result = rag.generate()
    
    eval_data.append(result)
    print(f"  answer: {result['answer'][:80]}...")
    print(f"  contexts: {len(result['contexts'])}\n")

print(f"\nall done ✅ total: {len(eval_data)}")

processing 1/10: Do mitochondria play a role in remodelling lace pl


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5114.47it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  answer: The role of mitochondria during PCD has been recognized in animals, however, it ...
  contexts: 1

processing 2/10: Landolt C and snellen e acuity: differences in str


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4521.26it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  answer: The mean difference between Landolt C acuity (LR) and Snellen E acuity (SE) was ...
  contexts: 3

processing 3/10: Syncope during bathing in infants, a pediatric for


...

  answer: Yes, this study evaluates the need for adjusting for reporting heterogeneity in ...
  contexts: 2

processing 9/10: Do mutations causing low HDL-C promote increased c


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7113.01it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  answer: Carotid intima-media thickness (cIMT) measurements were obtained in cases compri...
  contexts: 2

processing 10/10: A short stay or 23-hour ward in a general and acad


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3363.49it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  answer: The usefulness of a short stay or 23-hour ward was evaluated at Westmead Hospita...
  contexts: 3


all done ✅ total: 10


In [45]:
eval_data[1]

{'question': 'what role do mitochondria play in PCD?',
 'answer': 'The role of mitochondria during PCD has been recognized in animals, but it has been less studied during PCD in plants. The possible importance of mitochondrial permeability transition pore (PTP) formation during PCD was indirectly examined via in vivo cyclosporine A (CsA) treatment in the lace plant (Aponogeton madagascariensis).',
 'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.',
  'The following paper elucidates the role 

In [75]:
# CELL 4 — CHECK WHAT WE HAVE
import pandas as pd

eval_df = pd.DataFrame(eval_data)
display(eval_df.head())
print("\nshape:", eval_df.shape)
print("\ncolumns:", eval_df.columns.tolist())

,question,answer,contexts
0,Do mitochondria play a role in remodelling lac...,The role of mitochondria during PCD has been r...,[Programmed cell death (PCD) is the regulated ...
1,Landolt C and snellen e acuity: differences in...,The mean difference between Landolt C acuity (...,[Differences between Landolt C acuity (LR) and...
2,"Syncope during bathing in infants, a pediatric...",The context describes infants suffering from m...,[Eight infants aged 2 to 15 months were admitt...
3,Are the long-term results of the transanal pul...,This study examined the long-term outcome of T...,[The transanal endorectal pull-through (TERPT)...
4,Can tailored interventions increase mammograph...,The context discusses telephone counseling and...,"[Compared to usual care alone, telephone couns..."



shape: (10, 3)

columns: ['question', 'answer', 'contexts']


In [ ]:
### importing modules
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings
from datasets import Dataset

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5680\3761417559.py:5: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5680\3761417559.py:5: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5680\3761417559.py:5: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (


In [85]:
ragas_llm = LangchainLLMWrapper(
    Ollama(model="qwen3.5:2b")
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    OllamaEmbeddings(model="embeddinggemma")
)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5680\2559520668.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5680\2559520668.py:5: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


In [78]:
# ragas cannot handle empty context list
# remove rows with no contexts

filtered = [
    r for r in eval_data
    if len(r['contexts']) > 0
]
print(f"original : {len(eval_data)}")
print(f"filtered : {len(filtered)}")
print(f"removed  : {len(eval_data) - len(filtered)}")

original : 10
filtered : 10
removed  : 0


In [79]:
# CELL 8 — BUILD RAGAS DATASET
ragas_data = {
    "question": [r['question'] for r in filtered],
    "answer"  : [r['answer']   for r in filtered],
    "contexts": [r['contexts'] for r in filtered],
}

dataset = Dataset.from_dict(ragas_data)
print("dataset created ✅")
print(dataset)

dataset created ✅
Dataset({
    features: ['question', 'answer', 'contexts'],
    num_rows: 10
})


In [80]:
results = evaluate(
    dataset    = dataset,
    metrics    = [
        faithfulness,
        answer_relevancy,
    ],
    llm        = ragas_llm,
    embeddings = ragas_embeddings
) 

print("\nEVALUATION RESULTS")
print("─"*40)
print(f"faithfulness     : {results['faithfulness']:.4f}")
print(f"answer_relevancy : {results['answer_relevancy']:.4f}")
print(f"context_precision: {results['context_precision']:.4f}")

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]Exception raised in Job[2]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Evaluating:   5%|▌         | 1/20 [03:00<57:00, 180.03s/it]Exception raised in Job[4]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\asyncio\events.py", line 80,


EVALUATION RESULTS
────────────────────────────────────────


Task was destroyed but it is pending!
task: <Task pending name='Task-7492' coro=<Kernel.shell_main() running at C:\Users\ADMIN\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]>
c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\re\__init__.py:258: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  return pattern.translate(_special_chars_map)
Task was destroyed but it is pending!
task: <Task pending name='Task-7491' coro=<_async_in_context.<locals>.run_in_context() running at C:\Users\ADMIN\AppData\Roaming\Python\Python311\site-packages\ipykernel\utils.py:60> wait_for=<Task pending name='Task-7492' coro=<Kernel.shell_main() running at C:\Users\ADMIN\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\ADMIN\AppData\Roaming\Python\Python311\site-packages\zmq\eventloop\zmqstream.py:563]>


TypeError: unsupported format string passed to list.__format__

In [84]:
import ollama

models = ollama.list()
for m in models['models']:
    print(m['model'])

qwen3.5:2b
llama3.1:8b
qwen2.5:1.5b
embeddinggemma:latest
qwen3-embedding:0.6b
gemma4:latest


In [71]:
df.columns

Index(['pubid', 'question', 'context', 'long_answer', 'final_decision'], dtype='str')

### 2.3 Now let's compare with actual gt

In [81]:
gt_answers = df['long_answer'].tolist()[:10]

for idx in range(10):
    eval_data[idx]['reference'] = gt_answers[idx]

In [82]:
filtered_gt = [
    r for r in eval_data
    if len(r['contexts']) > 0
]
print(f"filtered: {len(filtered_gt)}")

filtered: 10


In [83]:
# CELL 4 — BUILD RAGAS DATASET WITH REFERENCE
from datasets import Dataset

ragas_data_gt = {
    "question" : [r['question']  for r in filtered_gt],
    "answer"   : [r['answer']    for r in filtered_gt],
    "contexts" : [r['contexts']  for r in filtered_gt],
    "reference": [r['reference'] for r in filtered_gt],
}

dataset_gt = Dataset.from_dict(ragas_data_gt)
print("dataset with GT created ✅")
print(dataset_gt)

dataset with GT created ✅
Dataset({
    features: ['question', 'answer', 'contexts', 'reference'],
    num_rows: 10
})


In [86]:
# CELL 5 — EVALUATE WITH ALL 4 METRICS
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)
from ragas.run_config import RunConfig

results_gt = evaluate(
    dataset    = dataset_gt,
    metrics    = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm        = ragas_llm,
    embeddings = ragas_embeddings,
    run_config = RunConfig(
        timeout    = 180,
        max_retries= 2,
        max_workers= 1
    )
)

print("\nBEFORE (no GT) vs AFTER (with GT)")
print("─"*45)
print(f"{'metric':<20} {'before':>8} {'after':>8}")
print("─"*45)
print(f"{'faithfulness':<20} {'0.8571':>8} {results_gt['faithfulness']:>8.4f}")
print(f"{'answer_relevancy':<20} {'0.5734':>8} {results_gt['answer_relevancy']:>8.4f}")
print(f"{'context_precision':<20} {'N/A':>8} {results_gt['context_precision']:>8.4f}")
print(f"{'context_recall':<20} {'N/A':>8} {results_gt['context_recall']:>8.4f}")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5680\942444711.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5680\942444711.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5680\942444711.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5680\942

KeyboardInterrupt: 